# Task 3 — Append YOLO26n Detections

Run YOLO26n inference on all images across all three datasets,
then append the results to the existing `get_detections_df()` DataFrame.

In [10]:
import os
os.chdir('..')

In [11]:
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
from utils import get_detections_df, get_image_file_names

In [12]:
IDOT_DIR   = Path("/Users/best/Desktop/IDOT")
MODEL_PATH = IDOT_DIR / "jupyter_notebooks/yolo26n.pt"
MODEL_LABEL = "yolo26n"
CONFIDENCE_THRESHOLD = 0.25

DATASETS = {
    "W042_08-09-2025": IDOT_DIR / "W042_08-09-2025",
    "W065_08-17-2025": IDOT_DIR / "W065_08-17-2025",
    "W06E_12-02-2025": IDOT_DIR / "W06E_12-02-2025",
}

print("Model exists:", MODEL_PATH.exists())
for name, path in DATASETS.items():
    print(f"{name} exists: {path.exists()}")

Model exists: True
W042_08-09-2025 exists: True
W065_08-17-2025 exists: True
W06E_12-02-2025 exists: True


## Step 1 — Load existing detections DataFrame

In [16]:
existing_dfs = []
for name, data_path in DATASETS.items():
    df = get_detections_df(data_path)
    df["dataset"] = name
    existing_dfs.append(df)

existing_df = pd.concat(existing_dfs, ignore_index=True)
print(f"Existing detections: {len(existing_df)} rows")
print(f"Models: {existing_df['model'].unique()}")
existing_df.head()

Existing detections: 628485 rows
Models: <StringArray>
['YOLOv5n', 'YOLOv10n', 'YOLOv8n']
Length: 3, dtype: str


,timestamp,timestamps_ns,has_image,model,class,confidence,x_min,y_min,x_max,y_max,dataset
0,2025-08-09T00:00:14.432149755+00:00,1754697614432149755,True,YOLOv5n,car,0.797410,294.247681,808.010437,556.992920,911.101013,W042_08-09-2025
1,2025-08-09T00:00:14.432149755+00:00,1754697614432149755,True,YOLOv5n,car,0.428542,1028.700439,668.934143,1127.017822,745.166199,W042_08-09-2025
2,2025-08-09T00:00:14.432149755+00:00,1754697614432149755,True,YOLOv5n,truck,0.358526,1365.500977,796.653687,1719.895752,910.816528,W042_08-09-2025
3,2025-08-09T00:00:14.432149755+00:00,1754697614432149755,True,YOLOv5n,car,0.305169,1125.280151,611.066406,1199.565552,654.421387,W042_08-09-2025
4,2025-08-09T00:00:14.432149755+00:00,1754697614432149755,True,YOLOv10n,car,0.852649,299.407593,802.812744,557.131897,912.103455,W042_08-09-2025


## Step 2 — Run YOLO26n on all images across all datasets

In [17]:
model = YOLO(str(MODEL_PATH))

yolo26n_rows = []

for dataset_name, data_path in DATASETS.items():
    image_path = data_path / "images"
    file_paths, file_names, timestamps_ns = get_image_file_names(image_path)

    print(f"\n=== {dataset_name} — {len(file_paths)} images ===")

    for file_path, timestamp_ns in zip(file_paths, timestamps_ns):
        results = model(str(file_path), conf=CONFIDENCE_THRESHOLD, verbose=False)
        result = results[0]

        class_names = result.names
        for box in result.boxes:
            x_min, y_min, x_max, y_max = box.xyxy[0].tolist()
            yolo26n_rows.append({
                "timestamp"    : None,
                "timestamps_ns": int(timestamp_ns),
                "has_image"    : True,
                "model"        : MODEL_LABEL,
                "class"        : class_names[int(box.cls[0])],
                "confidence"   : float(box.conf[0]),
                "x_min"        : x_min,
                "y_min"        : y_min,
                "x_max"        : x_max,
                "y_max"        : y_max,
                "dataset"      : dataset_name,
            })

yolo26n_df = pd.DataFrame(yolo26n_rows)
print(f"\nYOLO26n detections: {len(yolo26n_df)} rows")
print(f"Unique classes: {sorted(yolo26n_df['class'].unique())}")
yolo26n_df.head()


=== W042_08-09-2025 — 287 images ===

=== W065_08-17-2025 — 246 images ===

=== W06E_12-02-2025 — 287 images ===

YOLO26n detections: 3016 rows
Unique classes: ['bus', 'car', 'motorcycle', 'person', 'traffic light', 'train', 'truck']


,timestamp,timestamps_ns,has_image,model,class,confidence,x_min,y_min,x_max,y_max,dataset
0,None,1754697614432149755,True,yolo26n,car,0.842330,293.705994,802.391479,560.462952,914.354309,W042_08-09-2025
1,None,1754697614432149755,True,yolo26n,car,0.796858,1029.091675,674.554321,1126.587036,749.876526,W042_08-09-2025
2,None,1754697614432149755,True,yolo26n,car,0.612094,1365.777100,785.634460,1718.453247,911.769287,W042_08-09-2025
3,None,1754697614432149755,True,yolo26n,car,0.495358,1132.618286,613.135315,1201.352173,649.763062,W042_08-09-2025
4,None,1754697614432149755,True,yolo26n,car,0.374008,920.207336,608.476685,991.701904,651.317444,W042_08-09-2025


## Step 3 — Append YOLO26n results to existing DataFrame

In [18]:
combined_df = pd.concat([existing_df, yolo26n_df], ignore_index=True)

print(f"Combined detections: {len(combined_df)} rows")
print(f"Models: {combined_df['model'].unique()}")
combined_df.groupby(["dataset", "model"]).size().unstack(fill_value=0)

Combined detections: 631501 rows
Models: <StringArray>
['YOLOv5n', 'YOLOv10n', 'YOLOv8n', 'yolo26n']
Length: 4, dtype: str


model,YOLOv10n,YOLOv5n,YOLOv8n,yolo26n
dataset,,,,
W042_08-09-2025,63337,62683,81452,924
W065_08-17-2025,53872,47539,91152,1131
W06E_12-02-2025,71566,73951,82933,961


## Step 4 — Save combined DataFrame

In [21]:
out_path = Path("/Users/best/cities-in-motion-video/keira/combined_detections_with_yolo26n.csv")
combined_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to /Users/best/cities-in-motion-video/keira/combined_detections_with_yolo26n.csv


In [ ]:
yolo26n_out = Path("/Users/best/cities-in-motion-video/keira/yolo26n_detections.csv")
yolo26n_df.to_csv(yolo26n_out, index=False)
print(f"Saved YOLO26n detections to {yolo26n_out}")
print(f"Rows: {len(yolo26n_df)}")